# fr + rl · Lap2+Lap4 mapping (CLAHE + bright-blob filter) + Lap3 localization + se2 fusion

**Kernel:** `pixel-racers`

| Item | This notebook |
|---|---|
| Cameras | **`fr` / `rl` only** (no `r` map) |
| Preprocessing | After undistortion, **Lab-L CLAHE** for features; bright-blob keypoint filter on **gray0** |
| Features | `sp_nv_nn` (SuperPoint + NetVLAD + mutual NN) |
| Mapping | Lap **2+4**, $s$-ordered neighborhood |
| Query | Lap **3** |
| Fusion | `fuse_multi_camera_tables(..., ref_camera='fr', mode='se2')` |
| Output | `outputs_hfnet_clahe_fr_rl/`; after mapping, copy to `submission-demo/sfm-model/` |


In [ ]:
from __future__ import annotations
import importlib, json, os, sys, shutil
from pathlib import Path
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
ROOT = Path.cwd().resolve()
if ROOT.name == '3d-map-test':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / '3d-map-test'))
import sfm_experiment as sfm
import sfm_hfnet as hf
import image_preprocess as ipre
sfm = importlib.reload(sfm)
hf = importlib.reload(hf)
OUT = ROOT / '3d-map-test' / 'outputs_hfnet_clahe_fr_rl'
OUT.mkdir(parents=True, exist_ok=True)
POLYGONS = ROOT / 'fr-auto-mask' / 'polygons.json'
MAP_MCAP = ROOT.parent / 'c0_2026-10-13_22-53-42_camloc' / '23-00-38_cut_120m_update' / 'map.mcap'
SUBMIT_MAP = ROOT / 'submission-demo' / 'sfm-model'
CAMERAS = ['fr', 'rl']
MAP_LAPS = (2, 4)
QUERY_LAP = 3
REF_CAMERA = 'fr'
OVERRIDE_MAP = True
OVERRIDE_QUERY = True
device = hf.require_mps()
BACKEND, IS_PROXY = hf.resolve_hfnet_backend(None)
print('OUT', OUT)
print('preprocess', ipre.PREPROCESS_VERSION, ipre.DEFAULT_PHOTOMETRIC)
print('BACKEND', BACKEND, 'device', device)


In [ ]:
gt = sfm.load_gt()
laps_map = {L: sfm.load_lap(L) for L in MAP_LAPS}
lap_q = sfm.load_lap(QUERY_LAP)
calibs = {c: sfm.load_camera_calib(c) for c in CAMERAS}
track = hf.load_racing_line(MAP_MCAP)['track']
feats = hf.make_features('sp_nv_nn', max_keypoints=1024, resize=512, ratio=0.8)
runs = {}
for cam in CAMERAS:
    print('\n' + '='*60, 'CAMERA', cam)
    runs[cam] = hf.run_camera_multi_lap_s_pipeline(
        cam, feats=feats, calib=calibs[cam], gt=gt,
        laps_map=laps_map, lap_q=lap_q, track_line=track, out_root=OUT,
        map_laps=MAP_LAPS, query_lap=QUERY_LAP,
        map_max_frames=None, query_max_frames=None,
        map_stride=1, query_stride=1, min_nn_dist=0.50,
        neighbor_mode='arc', neighbor_k=6, neighbor_offset=3,
        n_max=None, min_baseline_m=0.5, epi_thresh_px=6.0, top_k=5,
        override_map=OVERRIDE_MAP, override_query=OVERRIDE_QUERY,
        polygons_path=POLYGONS, map_mcap=MAP_MCAP,
    )
    m = runs[cam]['metrics']
    print(f"[{cam}] A-RMSE={m['pos_rmse_xy_m']:.3f} points={runs[cam]['n_points']}")


In [ ]:
tables = {c: runs[c]['errors_df'] for c in CAMERAS}
fuse_dir = OUT / f'loc_lap{QUERY_LAP}_fused'
fuse_dir.mkdir(parents=True, exist_ok=True)
fused_se2 = hf.fuse_multi_camera_tables(tables, ref_camera=REF_CAMERA, mode='se2')
fused_se2.to_csv(fuse_dir / 'errors_se2.csv', index=False)
print('fuse success', fused_se2.ok.sum(), '/', len(fused_se2))
display(fused_se2.head())


In [ ]:
for cam in CAMERAS:
    src = OUT / f'map_{cam}_lap2+4'
    dst = SUBMIT_MAP / f'map_{cam}_lap2+4'
    if dst.is_dir():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print('copied', dst)
